# CPE393 Text Analytics — Homework: Extractive vs. Abstractive Summarization
**One-week group assignment (6 students)** | **BBC News full-text dataset** | **Submit a notebook and 10-minute group video**

### Goal
Compare **TextRank**, **LexRank**, and a **pretrained BART** summarizer on the **same three BBC News articles**, selected from **at least two categories**. Determine whether abstractive summarization necessarily gives better results.

**What is provided:** data-loading code, a guided TextRank skeleton, a LexRank hint, and complete BART inference code. **What you complete:** TextRank, LexRank, experiments, comparison, and explanations. You do not train BART.

**Important:** TextRank/LexRank select original sentences. BART may paraphrase or introduce unsupported claims. Compare actual summary lengths, not just requested lengths.

**Submission:** completed `.ipynb` (outputs visible, concise written answers), plus **10 min video** where *all* members explain a meaningful part; screen recording is sufficient, no slides required.

## 0. Setup and load BBC News (provided)
The original lab used the [BBC full-text dataset](http://mlg.ucd.ie/files/datasets/bbc-fulltext.zip), organized by category. We keep that dataset and fix a common preprocessing issue: **each file is an article, not a list of sentences**.

If the automatic download fails, upload `bbc-fulltext.zip` manually to the working directory. Do not use the built-in demonstration texts as your submitted BBC results.

In [ ]:
# Run once; installing may take a minute.
%pip -q install scikit-learn networkx nltk lexrank transformers torch sentencepiece pandas


In [ ]:
from pathlib import Path
from collections import Counter
import re, zipfile, urllib.request, random
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
nltk.download('punkt', quiet=True)
try:
    nltk.download('punkt_tab', quiet=True)
except Exception:
    pass
from nltk.tokenize import sent_tokenize

ZIP = Path('bbc-fulltext.zip')
if not ZIP.exists():
    try:
        urllib.request.urlretrieve('http://mlg.ucd.ie/files/datasets/bbc-fulltext.zip', ZIP)
    except Exception as exc:
        print('Automatic download failed; upload bbc-fulltext.zip manually:', exc)
if ZIP.exists() and not Path('bbc').exists():
    with zipfile.ZipFile(ZIP) as archive:
        archive.extractall('.')

root = Path('bbc')
articles = []
if root.exists():
    for category in ['business','entertainment','politics','sport','tech']:
        for file in sorted((root / category).glob('*.txt')):
            raw = file.read_text(encoding='latin-1').strip()
            if len(sent_tokenize(raw)) >= 8:
                articles.append({'category': category, 'file': file.name, 'text': raw})
print('BBC articles available:', len(articles))
print(Counter(a['category'] for a in articles))
assert articles, 'BBC data missing. Download/upload the dataset, then rerun.'


### Select three articles (student decision)
Choose **three BBC articles from at least two categories**. For reproducibility, record category, filename, sentence count, and the first 200 characters. Avoid exceptionally long articles on the first run. All methods must use **exactly the same articles**.

In [ ]:
# TODO: change the selection if you wish, but keep >=2 categories.
selected = [
    next(a for a in articles if a['category']=='business'),
    next(a for a in articles if a['category']=='politics'),
    next(a for a in articles if a['category']=='tech'),
]
for i, a in enumerate(selected, 1):
    print(i, a['category'], a['file'], 'sentences:', len(sent_tokenize(a['text'])))
    print(a['text'][:200].replace('\n',' '), '\n')
assert len(selected)==3 and len({a['category'] for a in selected})>=2


## 1. TextRank
You have already learned TextRank in the recorded lecture. Use TF-IDF cosine similarity to construct a **weighted sentence graph**, then apply PageRank. Extract the top-ranked sentences, **restoring original article order**.

**Important design choices:** exclude self-similarity; handle articles with few sentences; and use a common target of approximately **25–30% of the original sentence count** (not guaranteed identical word counts).

In [ ]:
def target_sentence_count(sentences, ratio=0.28):
    return max(1, min(len(sentences), round(len(sentences)*ratio)))

def textrank(text, ratio=0.28):
    sentences = sent_tokenize(text)
    if len(sentences) <= 1:
        return sentences
    tfidf = TfidfVectorizer(stop_words='english').fit_transform(sentences)
    similarity = cosine_similarity(tfidf)
    np.fill_diagonal(similarity, 0)
    # TODO 1: construct a weighted NetworkX graph from similarity
    graph = None
    # TODO 2: calculate PageRank scores for all sentence indices
    scores = None
    # TODO 3: choose top k indices, then put them back in original order
    indices = None
    return [sentences[i] for i in indices]

# Test on the first article
# print(' '.join(textrank(selected[0]['text'])))


**Checkpoint:** Explain in 2–3 sentences why the similarity matrix has zeros on the diagonal and why extracted sentences must be restored to article order.

## 2. LexRank
Use the `lexrank` library. It requires a **corpus of tokenized sentences** to estimate IDF. Build the corpus from BBC articles (e.g., the selected articles or a fixed, documented larger subset). Then summarize the **same three articles**. The library's `get_summary` selects sentences; restore their original order if needed.

**Experiment:** For one article compare **thresholded LexRank** (`threshold=0.25`) with **continuous LexRank** (`threshold=None`). What changes?

In [ ]:
from lexrank import STOPWORDS, LexRank

# Hint:
# corpus = [sent_tokenize(a['text']) for a in articles[:200]]
# lxr = LexRank(corpus, stopwords=STOPWORDS['en'])
# lxr.get_summary(sentences, summary_size=k, threshold=0.25)

# TODO 4: construct a LexRank object using a documented corpus.
# TODO 5: implement lexrank_summary(text, ratio=0.28, threshold=0.25).
#           Return selected sentences in source order.
# TODO 6: compare threshold=0.25 and threshold=None for one article.


## 3. BART
This is **abstractive** summarization using a pretrained model. You are **not** expected to implement Transformers or train a model. The model `facebook/bart-large-cnn` is fine-tuned on CNN/DailyMail, not BBC; discuss this limitation. A GPU is helpful, but CPU can work more slowly.

BART has a limited input length. We explicitly truncate overly long articles; check whether truncation occurs and discuss its impact. Output lengths are token limits, **not** a guaranteed 25–30% compression ratio.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = 'facebook/bart-large-cnn'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device).eval()
print('Device:', device)

def bart_summary(text, max_new_tokens=120, min_new_tokens=35):
    encoded_full = tokenizer(text, truncation=False, add_special_tokens=True)
    was_truncated = len(encoded_full['input_ids']) > 1024
    inputs = tokenizer(text, max_length=1024, truncation=True, return_tensors='pt').to(device)
    with torch.no_grad():
        ids = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             min_new_tokens=min_new_tokens,
                             num_beams=4, do_sample=False)
    return tokenizer.decode(ids[0], skip_special_tokens=True), was_truncated

# First article demonstration. If downloads or memory fail, use Colab GPU or retry.
example_bart, truncated = bart_summary(selected[0]['text'])
print('Truncated input?', truncated)
print(example_bart)


## 4. Apply all three methods to all three articles
Fill in the missing cells. Use a similar **intended** summary length for the extractive methods, but record actual **word counts** for every output. BART's output length may differ substantially; don't quietly claim that lengths are matched.

Store the outputs in a dictionary keyed by `(article_number, method)` so you can compare them later.

In [ ]:
results = {}
for i, article in enumerate(selected, start=1):
    text = article['text']
    # TODO 7: run TextRank and LexRank; join returned sentences with spaces.
    # TODO 8: run BART; record whether its input was truncated.
    # results[(i, 'TextRank')] = ...
    # results[(i, 'LexRank')] = ...
    # results[(i, 'BART')] = ...

# TODO 9: create a DataFrame with article, method, original words,
#          summary words, compression ratio = summary words / original words.


## 5. Compare results
For **each article**, inspect all three summaries and complete a concise comparison. Support your judgment with **specific examples** from the original text and outputs.

Evaluate:
1. **Coverage:** Are the key facts and events represented?
2. **Readability/coherence:** Does the summary flow naturally?
3. **Redundancy:** Is information unnecessarily repeated?
4. **Faithfulness:** Any invented or distorted facts, dates, quantities, or people?
5. **Abstraction:** Are sentences copied, reworded, or combined?

Use a simple 1–5 scale for **coverage, readability, faithfulness**, and give **one concrete example** per article. Do not treat the scores as objective ground truth. No ROUGE required because the BBC full-text dataset here does not provide paired gold summaries.

In [ ]:
# TODO 10: display each article and all three summaries.
# Suggested pattern:
# for i, a in enumerate(selected, 1):
#     print('ARTICLE', i, a['file'])
#     for method in ['TextRank','LexRank','BART']:
#         print(method, results[(i, method)])

# TODO 11: create a comparison table with 1–5 ratings and evidence.


### Reflection (answer in Markdown)
**Q1.** On which article do TextRank and LexRank select notably different information? Explain using the methods' similarities/differences.

**Q2.** Give an example where BART paraphrases well **or** changes a fact. Quote only a short relevant span and verify against the source.

**Q3.** Is BART always better? Make a reasoned claim based on your three articles. Consider input truncation and unequal summary lengths.

**Q4.** What are two limitations of this comparison (sample size, subjective ratings, model domain, summary length, etc.)?

**Q5.** What is the difference between sentence compression and full-document abstractive summarization? (Conceptual question; no parse-tree implementation required.)

## 6. Video and submission checklist
**Video (6–8 minutes; all 6 students must speak):**
- Briefly introduce the selected BBC articles and why they were chosen.
- Explain how TextRank and LexRank differ, and demonstrate at least one output.
- Explain that BART is pretrained and how you used it (no training).
- Show **one meaningful comparison**, including evidence about factual accuracy.
- End with the group's answer to: *Does abstractive summarization necessarily outperform extractive summarization?*

**Submit:** one completed notebook (including executed outputs and reflections), one video link/file accessible to the instructor. No slide deck required. **Group members should each explain a substantive contribution.**

### Rubric (100 points)
TextRank implementation **15**; LexRank implementation and threshold investigation **15**; BART inference and reporting **10**; consistent experiments and length reporting **15**; qualitative comparison with evidence **25**; reflection/limitations **10**; group video and participation **10**.
